In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It installs AaronTools and embeds the required Gaussian
# output file (water.log) directly — no external downloads needed.
#
# numpy and matplotlib are pre-installed in Google Colab.
# AaronTools will be installed below (takes ~30 seconds on first run).

import sys, os, subprocess, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")

# ── Install AaronTools ────────────────────────────────────────────────────────
print("\nInstalling AaronTools...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "AaronTools"])
print("AaronTools installed ✓")

# ── Embed water.log ───────────────────────────────────────────────────────────
# B3LYP/6-31G* geometry optimization and harmonic frequency calculation for H2O
_WATER_LOG_CONTENT = """
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

if not os.path.exists("water.log"):
    with open("water.log", "w") as _f:
        _f.write(_WATER_LOG_CONTENT)
    print("Written : water.log")
else:
    print("Found existing: water.log")

print("\nEnvironment ready ✓")


# Python for Chemists — Part 3: Structure Alignment with AaronTools
## Demonstration Walkthrough

This notebook is a **guided demonstration** of molecular structure alignment using AaronTools. Each section introduces a concept, shows the working code, and explains the thought process and chemistry behind it. Run the cells top to bottom to follow along.

**What this notebook covers**

| Section | Topic |
|---------|-------|
| 1 | Installation and imports |
| 2 | Reading molecular structures |
| 3 | Inspecting a geometry |
| 4 | Center of mass and coordinate manipulation |
| 5 | RMSD without alignment |
| 6 | RMSD with alignment — the Kabsch algorithm |
| 7 | Heavy-atom RMSD |
| 8 | Targeted alignment on a fragment |
| 9 | RMSD_permute for symmetric molecules |
| 10 | Filtering a conformer ensemble |
| 11 | Connecting to Gaussian: geometry from a `.log` file |

---
## Section 1 — Installation and Imports

AaronTools is installed with pip:

```bash
pip install AaronTools
```

The two classes you will use throughout this notebook are:
- **`Geometry`** — the central object representing a molecular structure (atoms + coordinates + connectivity)
- **`FileReader`** — a lower-level file parser; `Geometry` can accept one directly, but you will rarely need to use `FileReader` explicitly

AaronTools uses numpy arrays internally, so numpy will also be imported.

In [ ]:
from AaronTools.geometry import Geometry
from AaronTools.fileIO import FileReader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import AaronTools
print("AaronTools version:", AaronTools.__version__)
print("All imports successful!")

---
## Section 2 — Reading Molecular Structures

AaronTools can read structures from many file formats that you will encounter in a QM lab: Gaussian `.log` output files, Gaussian `.com`/`.gjf` input files, `.xyz` files, `.pdb` files, and more.

The simplest way to load a structure is to pass a filename directly to `Geometry`:

```python
mol = Geometry("molecule.xyz")       # from XYZ file
mol = Geometry("opt.log")            # from Gaussian output
mol = Geometry("molecule.com")       # from Gaussian input
```

When reading a Gaussian `.log` file, AaronTools extracts the **last** Standard Orientation block — which corresponds to the final optimized geometry if the job converged.

The setup cell below writes the sample `.xyz` files we will use throughout this notebook — ethane in its **staggered** (ground state) and **eclipsed** (transition state) conformations. These are the two limiting geometries of the simplest C−C internal rotation.

In [ ]:
# Ethane staggered (H-C-C-H dihedral = 60°) — the ground state
staggered_xyz = """8
Ethane staggered conformer
C   0.000   0.000   0.765
C   0.000   0.000  -0.765
H   1.025   0.000   1.157
H  -0.512   0.887   1.157
H  -0.512  -0.887   1.157
H   0.512   0.887  -1.157
H  -1.025   0.000  -1.157
H   0.512  -0.887  -1.157
"""

# Ethane eclipsed (H-C-C-H dihedral = 0°) — the rotational transition state
eclipsed_xyz = """8
Ethane eclipsed conformer
C   0.000   0.000   0.765
C   0.000   0.000  -0.765
H   1.025   0.000   1.157
H  -0.512   0.887   1.157
H  -0.512  -0.887   1.157
H   1.025   0.000  -1.157
H  -0.512   0.887  -1.157
H  -0.512  -0.887  -1.157
"""

# Ethane randomly displaced — same staggered geometry but translated and rotated
# (as if two Gaussian jobs placed the molecule in different spots)
displaced_xyz = """8
Ethane staggered displaced
C   3.412   1.765   0.218
C   2.838   1.068  -0.987
H   4.346   1.293   0.534
H   3.587   2.791   0.015
H   2.695   1.808   1.094
H   1.900   1.538  -1.302
H   2.660   0.042  -0.790
H   3.580   1.064  -1.783
"""

with open("ethane_staggered.xyz", "w") as f:
    f.write(staggered_xyz)
with open("ethane_eclipsed.xyz", "w") as f:
    f.write(eclipsed_xyz)
with open("ethane_displaced.xyz", "w") as f:
    f.write(displaced_xyz)

print("Sample XYZ files written.")

In [ ]:
staggered  = Geometry("ethane_staggered.xyz")
eclipsed   = Geometry("ethane_eclipsed.xyz")
displaced  = Geometry("ethane_displaced.xyz")

print(f"Staggered : {staggered.n_atoms} atoms, name = '{staggered.name}'")
print(f"Eclipsed  : {eclipsed.n_atoms} atoms")
print(f"Displaced : {displaced.n_atoms} atoms")

The `Geometry` constructor detects the file format from the extension and hands off to the appropriate reader in `fileIO.py`. The `.name` attribute is taken from the comment line of an XYZ file (the second line) or from the filename stem for other formats. Checking `n_atoms` immediately after loading is a good habit — it confirms you got the structure you expected before any analysis begins.

---
## Section 3 — Inspecting a Geometry Object

A `Geometry` object exposes the molecular structure through several attributes:

| Attribute | Type | Contents |
|-----------|------|----------|
| `.atoms` | `list` of `Atom` | All atom objects in order |
| `.n_atoms` | `int` | Number of atoms |
| `.coords` | `(N, 3) ndarray` | All coordinates in Å |
| `.name` | `str` | Molecule name (from comment line in XYZ, or filename) |

Each `Atom` object has:

| Attribute | Type | Contents |
|-----------|------|----------|
| `.element` | `str` | Element symbol, e.g. `"C"`, `"H"`, `"N"` |
| `.coords` | `(3,) ndarray` | xyz coordinates of this atom in Å |
| `.name` | `str` | Atom index label |

The `.coords` array on the `Geometry` object is a **copy** of the individual atom coordinates stacked into a matrix — modifying it does not move the atoms. Use `coord_shift` or `rotate` (Section 4) for that.

In [ ]:
mol = Geometry("ethane_staggered.xyz")

print(f"Molecule   : {mol.name}")
print(f"Atoms      : {mol.n_atoms}")
print(f"Coord shape: {mol.coords.shape}")
print()
print("  idx  element       x          y          z")
print("  ─────────────────────────────────────────")
for i, atom in enumerate(mol.atoms):
    x, y, z = atom.coords
    print(f"  {i:3d}    {atom.element:2s}     {x:8.3f}   {y:8.3f}   {z:8.3f}")

The `for atom in mol.atoms` loop is the primary way to inspect individual atoms. The `atom.element` attribute returns a capitalised symbol string — `"C"`, `"H"`, `"N"`, etc. — which you can compare directly against strings in selection logic. The `atom.coords` attribute is a 1-D numpy array of length 3; arithmetic on it follows normal numpy broadcasting rules.

---
## Section 4 — Center of Mass and Coordinate Manipulation

Before computing RMSD you often need to **center** a molecule at the origin so that translation does not contribute to the comparison. AaronTools provides:

```python
com = mol.COM(mass_weight=True)   # mass-weighted center of mass (x, y, z)
com = mol.COM(mass_weight=False)  # geometric centroid (unweighted average)
```

To translate the molecule, pass a displacement vector to `coord_shift`:
```python
mol.coord_shift([dx, dy, dz])     # shifts ALL atoms by (dx, dy, dz) — modifies in place
```

To rotate the molecule around an axis passing through the origin:
```python
mol.rotate([vx, vy, vz], angle)   # angle in degrees, axis = (vx, vy, vz)
```

> **Important:** both `coord_shift` and `rotate` modify the geometry **in place**. If you want to preserve the original, use `mol.copy()` first.

In [ ]:
mol = Geometry("ethane_staggered.xyz")

com_before = mol.COM(mass_weight=True)
print(f"COM before centering: ({com_before[0]:.6f}, {com_before[1]:.6f}, {com_before[2]:.6f})")

# Shift the molecule so the COM lands at the origin
mol.coord_shift(-com_before)

com_after = mol.COM(mass_weight=True)
print(f"COM after centering : ({com_after[0]:.2e}, {com_after[1]:.2e}, {com_after[2]:.2e})")
print("(values ~0 confirm the molecule is centred at the origin)")

Centering at the origin is almost always the first step before any RMSD comparison. The `COM()` method returns a numpy array, so passing it as `-com_before` directly to `coord_shift` negates the vector and shifts the molecule to the origin. The near-zero values printed after centering confirm the operation succeeded — floating-point rounding keeps the residual below 10⁻¹⁵ Å.

---
## Section 5 — RMSD Without Alignment

The RMSD between two structures with the **same atom ordering** is:

$$\text{RMSD} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}|\vec{r}_i - \vec{r}_i^{\,\text{ref}}|^2}$$

In AaronTools:
```python
val = mol.RMSD(ref)   # no alignment — measures raw positional difference
```

Without alignment, even an identical molecule placed 5 Å away will report a large RMSD. This is generally not useful for comparing molecular shapes — but it is useful as a quick sanity check on whether two files truly represent the same geometry.

In [ ]:
ref   = Geometry("ethane_staggered.xyz")
ecl   = Geometry("ethane_eclipsed.xyz")
disp  = Geometry("ethane_displaced.xyz")   # same geometry, different position

rmsd_ecl  = ref.RMSD(ecl)
rmsd_disp = ref.RMSD(disp)

print(f"RMSD(staggered, eclipsed) — no align : {rmsd_ecl:.4f} Å")
print(f"RMSD(staggered, displaced) — no align: {rmsd_disp:.4f} Å")
print()
print("The displaced structure is IDENTICAL to staggered but looks very different")
print("without alignment because it lives in a different part of space.")

The raw RMSD without alignment is rarely chemically informative, but it is useful for one specific purpose: detecting whether two files are *literally the same set of coordinates*. If the unaligned RMSD is already zero, no alignment is needed. The large RMSD for the displaced structure shows that placing the same molecule at a different position in space — as Gaussian routinely does for different jobs — completely dominates the metric without prior alignment.

---
## Section 6 — RMSD With Alignment: The Kabsch Algorithm

To get a meaningful structural comparison, we need to find the rotation and translation that **minimise** the RMSD before measuring it. The standard algorithm for this is the **Kabsch algorithm** (1976), which uses Singular Value Decomposition (SVD) to find the optimal rotation matrix in closed form.

The steps are:
1. Translate both structures so their centers of mass coincide at the origin.
2. Compute the covariance matrix $H = P^T Q$ where $P$ and $Q$ are the coordinate matrices of the two structures.
3. Decompose $H = U \Sigma V^T$ via SVD.
4. The optimal rotation matrix is $R = V U^T$ (with a sign correction if $\det(R) = -1$).
5. Apply $R$ to one structure and compute the RMSD.

AaronTools does all of this internally when you pass `align=True`:

```python
rmsd_val = mol.RMSD(ref, align=True)
```

**`align=True` modifies `mol` in place** — after the call, `mol.coords` contains the optimally rotated and translated coordinates. The return value is the minimised RMSD.

In [ ]:
ref  = Geometry("ethane_staggered.xyz")
ecl  = Geometry("ethane_eclipsed.xyz").copy()   # copy so we can align it
disp = Geometry("ethane_displaced.xyz").copy()

rmsd_ecl_before  = ref.RMSD(ecl)
rmsd_disp_before = ref.RMSD(disp)

# Now align and measure
rmsd_ecl_after  = ecl.RMSD(ref,  align=True)
rmsd_disp_after = disp.RMSD(ref, align=True)

print("                              Before align    After align")
print(f"  staggered vs eclipsed  :    {rmsd_ecl_before:8.4f} Å     {rmsd_ecl_after:8.4f} Å")
print(f"  staggered vs displaced :    {rmsd_disp_before:8.4f} Å     {rmsd_disp_after:8.4f} Å")
print()
print("After alignment the displaced structure (identical geometry, different")
print("position) gives RMSD ≈ 0.  The eclipsed conformer gives a non-zero RMSD")
print("because the H atoms are genuinely in different positions.")

Notice the two-step pattern: compute the RMSD before alignment (for reference), then compute it again with `align=True`. Because `align=True` modifies `ecl` and `disp` in place, fresh copies are needed for the second call. After alignment the displaced structure gives RMSD ≈ 0 (confirming identity) while the eclipsed conformer gives a non-zero value that reflects the genuine change in hydrogen positions — about 0.4–0.5 Å for the 60° torsional difference in ethane.

---
## Section 7 — Heavy-Atom RMSD

Hydrogen positions are often less reliable in computational chemistry:
- X-ray crystallography determines H positions poorly (X-rays scatter from electrons, and H has only one).
- Many force fields and semiempirical methods are less accurate for H than for heavy atoms.
- Freely rotating methyl groups can produce large RMSD values that do not reflect meaningful structural differences.

It is therefore common to report **heavy-atom RMSD** (all non-hydrogen atoms) alongside or instead of all-atom RMSD.

```python
rmsd_heavy = mol.RMSD(ref, align=True, heavy_only=True)
```

In [ ]:
ref = Geometry("ethane_staggered.xyz")
ecl = Geometry("ethane_eclipsed.xyz").copy()

rmsd_all   = ecl.RMSD(ref, align=True)

# Reload eclipsed for a fresh unaligned copy for the heavy-only comparison
ecl2 = Geometry("ethane_eclipsed.xyz").copy()
rmsd_heavy = ecl2.RMSD(ref, align=True, heavy_only=True)

print(f"All-atom RMSD  (staggered vs eclipsed): {rmsd_all:.4f} Å")
print(f"Heavy-atom RMSD(staggered vs eclipsed): {rmsd_heavy:.4f} Å")
print()
print("For ethane the only heavy atoms are the two carbons, which don't move")
print("between conformers — so the heavy-atom RMSD is ~0 even though the")
print("hydrogen positions differ substantially.")

Heavy-atom RMSD is particularly useful when comparing computed structures to X-ray data, because crystallography determines heavy-atom positions precisely but H positions are unreliable. For ethane specifically the two carbons sit on the C₂ symmetry axis in both conformers, so the carbon skeleton is identical and the heavy-atom RMSD is zero regardless of whether the H atoms are staggered or eclipsed.

---
## Section 8 — Targeted Alignment: Aligning on a Fragment

Sometimes you want to align only on a **subset** of atoms — for example, aligning two molecules on their shared scaffold while letting the rest of the structure float. AaronTools supports this via the `targets` and `ref_targets` parameters:

```python
rmsd = mol.RMSD(
    ref,
    align=True,
    targets="1,2",      # atom indices (1-based) in mol used for fitting
    ref_targets="1,2",  # corresponding atom indices in ref
)
```

You can also specify elements: `targets="C"` selects all carbon atoms. After the call, the **whole molecule** has been rotated/translated to minimise the RMSD over the target atoms — but the RMSD value returned is computed over all atoms unless you also restrict with `heavy_only`.

A common use case: aligning two structures on just the carbon skeleton, then examining where the functional groups end up.

In [ ]:
ref = Geometry("ethane_staggered.xyz")
ecl = Geometry("ethane_eclipsed.xyz").copy()

# Align eclipsed to staggered using ONLY the two carbon atoms (indices 1 and 2, 1-based)
rmsd_c_targeted = ecl.RMSD(
    ref,
    align=True,
    targets="C",      # use carbon atoms in ecl for the fit
    ref_targets="C"   # and the corresponding carbons in ref
)

print(f"RMSD after carbon-targeted alignment: {rmsd_c_targeted:.4f} Å")
print("(This aligns the C-C axis; H atoms are placed wherever the rotation puts them)")

The `targets` parameter accepts an element symbol string (`"C"`), a comma-separated index list (`"1,2"`), or a range string (`"1-4"`). When you specify targets, the Kabsch rotation is solved using only those atoms, but the resulting transformation is then applied to the entire molecule. The RMSD value returned is still computed over all atoms (unless `heavy_only=True` is also set), so it reflects how well the rest of the structure follows after the targeted fit.

---
## Section 9 — RMSD_permute: Handling Symmetric Molecules

Consider a molecule with several identical atoms — ammonia (NH₃), benzene (C₆H₆), or even just two ends of ethane. Depending on how the atoms happen to be numbered in two different files, a standard `RMSD` call may pair H atom 1 in structure A with H atom 2 in structure B and report a misleadingly large RMSD for what are actually identical structures.

`RMSD_permute` fixes this by trying **all valid permutations** of same-element atoms and returning the minimum RMSD found:

```python
rmsd = mol.RMSD_permute(ref, align=True)
```

This is more expensive than a single `RMSD` call — the cost grows factorially with the number of identical atoms — but for small molecules like peptide fragments or simple organics it is typically fast.

In [ ]:
# Create a version of staggered ethane where the H atoms on C1 are renumbered
# (swap H3 and H4) — standard RMSD will see a 'difference' that is not real.

import copy as py_copy

ref  = Geometry("ethane_staggered.xyz")
perm = Geometry("ethane_staggered.xyz").copy()

# Manually swap the coords of H atoms at index 2 and 3 (0-based)
coords_2 = perm.atoms[2].coords.copy()
perm.atoms[2].coords = perm.atoms[3].coords.copy()
perm.atoms[3].coords = coords_2

rmsd_std  = perm.copy().RMSD(ref,         align=True)
rmsd_perm = perm.copy().RMSD_permute(ref, align=True)

print(f"Standard RMSD after swapping two H labels: {rmsd_std:.4f} Å")
print(f"RMSD_permute after swapping two H labels : {rmsd_perm:.4f} Å")
print()
print("RMSD_permute correctly identifies these as the same structure (RMSD ≈ 0)")
print("by trying the permutation that maps H3→H4 and H4→H3.")

The key insight demonstrated here is that `RMSD` and `RMSD_permute` can give very different answers for the same two structures when atom numbering is inconsistent. In a conformer search, different optimisation trajectories will generally produce structures with the same connectivity but arbitrary atom ordering within each element. `RMSD_permute` handles this correctly at the cost of a factorial search over same-element atoms — acceptable for small molecules, but prohibitively expensive for large ones with many identical atoms.

---
## Section 10 — Filtering a Conformer Ensemble

Conformer search algorithms (and grid scans of torsion angles) produce many structures, often with redundant near-duplicates. A standard post-processing step is to cluster structures by RMSD and keep only one representative from each cluster — the one with the lowest energy.

The simplest version of this is **greedy RMSD filtering**: iterate through the list of conformers in energy order; if a conformer has RMSD > threshold from all previously accepted structures, accept it; otherwise discard it.

```python
THRESHOLD = 0.25   # Å — structures closer than this are considered duplicates
```

The cell below creates a small synthetic conformer set to demonstrate the approach.

In [ ]:
import os, textwrap

# Build a synthetic ensemble: copies of staggered and eclipsed with small noise added
rng = np.random.default_rng(42)
conformer_files = []
template_s = Geometry("ethane_staggered.xyz")
template_e = Geometry("ethane_eclipsed.xyz")

for i in range(6):
    base = template_s if i < 3 else template_e
    conf = base.copy()
    # add tiny noise (< 0.05 Å) so they are near-duplicates, not exact copies
    for atom in conf.atoms:
        atom.coords += rng.normal(0, 0.02, 3)
    fname = f"conformer_{i}.xyz"
    conf.write(fname)
    conformer_files.append(fname)

# Greedy filtering
THRESHOLD = 0.25   # Å
accepted = []

for fname in conformer_files:
    candidate = Geometry(fname)
    is_duplicate = False
    for ref_geom in accepted:
        test_copy = candidate.copy()
        r = test_copy.RMSD(ref_geom.copy(), align=True)
        if r < THRESHOLD:
            is_duplicate = True
            break
    if not is_duplicate:
        accepted.append(candidate)
        print(f"  ACCEPTED: {fname}")
    else:
        print(f"  rejected: {fname} (duplicate)")

print(f"\n{len(accepted)} unique structures kept from {len(conformer_files)} candidates.")

The greedy filter is $O(N^2)$ in the number of conformers and $O(N_\text{atoms})$ per RMSD call, so it scales well for ensembles of hundreds of structures. The threshold choice involves a trade-off: too tight and near-identical structures from numerical noise in the optimiser get kept as false positives; too loose and genuinely distinct low-energy conformers get discarded. Values between 0.2 and 0.5 Å are common in the literature for small organic molecules.

---
## Section 11 — Connecting to Gaussian: Geometry from a `.log` File

In Parts 1 and 2 you wrote a custom Gaussian parser from scratch. AaronTools can do the same job in one line — and it will handle edge cases (multiple optimisation steps, failed jobs, frequency calculations) automatically.

For a completed geometry optimisation, `Geometry("job.log")` loads the **final Standard orientation** from the log file. This is the optimised structure.

Suppose you want to compare the geometry your custom parser extracted to what AaronTools reads — a useful cross-check. Or suppose you have a reference structure from the Cambridge Structural Database in `.xyz` format and want to know how well your DFT optimisation reproduced it.

In [ ]:
# Uses the water.log file from the Track 1 folder
import os
log_path = os.path.join("..", "water.log") if os.path.exists("../water.log") else "water.log"

water = Geometry(log_path)

print(f"Atoms loaded from water.log: {water.n_atoms}")
print(f"Formula: H={sum(1 for a in water.atoms if a.element=='H')} "
      f"O={sum(1 for a in water.atoms if a.element=='O')}")
print()
print("Atomic positions (Å):")
for atom in water.atoms:
    print(f"  {atom.element:2s}  {atom.coords[0]:9.5f}  {atom.coords[1]:9.5f}  {atom.coords[2]:9.5f}")

Reading directly from a Gaussian `.log` file connects AaronTools seamlessly to the output files you already know how to generate. AaronTools extracts the final Standard orientation automatically, so you get the converged geometry without any manual parsing. The O−H distances and H−O−H angle printed here should match the values your Part 2 parser extracted from the same file — a useful cross-check that both approaches are correct.

---
## Summary

You now have a working toolkit for structure alignment with AaronTools:

- **Loading structures** with `Geometry(filename)` from `.xyz`, `.log`, `.com`, and other formats
- **Inspecting geometries** via `.atoms`, `.coords`, `.n_atoms`, and `Atom.element` / `Atom.coords`
- **Centring molecules** with `COM()` and `coord_shift()`
- **Measuring RMSD** with and without alignment via `RMSD(ref, align=False/True)`
- **Heavy-atom RMSD** with `heavy_only=True`
- **Targeted alignment** on a subset of atoms with `targets=` and `ref_targets=`
- **Permutation-aware RMSD** with `RMSD_permute()` for symmetric molecules
- **Conformer filtering** using greedy pairwise RMSD comparison
- **Reading Gaussian output** directly into a `Geometry` object

**Where to go next:**
- Use `Geometry.write("output.com")` to write AaronTools-manipulated structures back to Gaussian input format for further calculations.
- Explore `AaronTools.substituent` for swapping functional groups on a scaffold.
- Look at `AaronTools.component` for building transition-metal complexes.
- Combine with Part 2's parser: extract a series of geometries along an IRC and plot RMSD from the reactant as a function of the reaction coordinate.